In [291]:
import pandas as pd
from qiskit.quantum_info import SparsePauliOp, Statevector, Pauli
from qiskit.circuit.library import PauliEvolutionGate
import numpy as np
import multiprocessing as mp
from dataclasses import dataclass
import numpy as np
from te_pai import pai, sampling
from qulacs import QuantumCircuit, Observable, QuantumState
from qulacs.gate import  PauliRotation
from scipy.sparse.linalg import expm_multiply
import multiprocessing as mp
from dataclasses import dataclass
from functools import partial
import numpy as np
from te_pai import pai, sampling
from te_pai.backend import Simulator
from scipy.stats import binom

In [292]:
# Import Hamiltonian data
# This is rightmost is qubit 0 (LSB)
df = pd.read_csv("ES_H4_linear_R1.2_sto-6g.csv")
coeffs = df["coef"].astype(float).tolist()
paulis = df["label"].tolist()
nq = len(paulis[0]) if paulis else 0

In [308]:
# initial Hartree-Fock state
psi0 = QuantumState(nq)
psi0.set_computational_basis(int("00001111", 2))
T = 1.0
n_snap = 10

In [235]:
def pauli_to_qulacs_str(label: str) -> str:
    terms = []
    for q in range(nq):
        p = label[nq-1-q]
        if p != "I":
            terms.append(f"{p} {q}")
    return " ".join(terms)

def pauli_to_qulacs_ind(label):
    n = len(label)
    idx, ids = [], []
    for q in range(n):
        p = label[n-1-q]
        if p != "I":
            idx.append(q)
            ids.append({"I":0, "X":1, "Y":2, "Z":3}[p])
    return idx, ids

H = Observable(nq)
for lab, c in zip(paulis, coeffs):
    H.add_operator(c, pauli_to_qulacs_str(lab))

In [309]:
Hmat = H.get_matrix()
psiT = expm_multiply((-1j*T) * Hmat, psi0.get_vector()) 

def z_expect_from_vec(vec, q):
    prob = np.abs(vec)**2
    bits = (np.arange(prob.size) >> q) & 1  
    return float(np.sum((1 - 2*bits) * prob))

def site_occupations_from_vec(vec, n_qubits):
    occs = []
    for i in range(n_qubits // 2):
        q_up, q_dn = 2*i, 2*i+1
        n_up = 0.5 * (1 - z_expect_from_vec(vec, q_up))
        n_dn = 0.5 * (1 - z_expect_from_vec(vec, q_dn))
        occs.append(n_up + n_dn)
    return occs

occs = site_occupations_from_vec(psiT, nq)
for i, ni in enumerate(occs):
    print(f"site {i}: n_i = {ni:.6f}")

site 0: n_i = 1.930536
site 1: n_i = 1.912186
site 2: n_i = 0.089344
site 3: n_i = 0.067935


In [314]:
N = 20
def build_trotter_circuit(labels, coeffs, T, N):
    dt = T / N
    circ = QuantumCircuit(len(labels[0]))
    for _ in range(N):
        for lab, c in zip(labels, coeffs):
            idx, ids = pauli_to_qulacs_ind(lab)
            if not idx:
                continue
            angle = 2.0 * dt * c
            circ.add_gate(PauliRotation(idx, ids, angle))
    return circ

def prob(state, q):
    obs = Observable(nq)
    obs.add_operator(1.0, f"Z {q}")
    return float(((obs.get_expectation_value(state)+1)/2).real)

def site_occupations_qulacs(state):
    occs = []
    for i in range(nq // 2):
        q_up, q_dn = 2*i, 2*i+1
        n_up = 0.5 * (1 -(2* prob(state, q_up)-1))
        n_dn = 0.5 * (1 - (2*prob(state, q_dn)-1))
        occs.append(n_up + n_dn)
    return occs

U_circ = build_trotter_circuit(paulis, coeffs, T, N)
psi = QuantumState(nq)
psi.set_computational_basis(int("00001111", 2))
U_circ.update_quantum_state(psi)
occs = site_occupations_qulacs(psi)
for i, ni in enumerate(occs):
    print(f"site {i}: n_i = {ni:.6f}")

site 0: n_i = 1.930541
site 1: n_i = 1.912161
site 2: n_i = 0.089355
site 3: n_i = 0.067943


In [ ]:
N = 100
Δ   = np.pi / (2**8)
L   = len(paulis)
n_circ = 100
def l1_norm_over_time(T: float):
    return T * float(np.sum(np.abs(coeffs)))
steps  = np.linspace(0, T, N, endpoint=False)
angles = [2.0 * abs(c) * T / N for c in coeffs]
terms  = [pauli_to_qulacs_ind(lab) for lab in paulis]
probs  = pai.prob_list(angles, Δ)
n = int(N / n_snap) 
gam_list = [1] + [pai.gamma(angles, Δ) ** ((i + 1) * n) for i in range(n_snap)]
gamma = gam_list[-1] if N > 0 else 0.0
overhead = np.exp(2.0 * l1_norm_over_time(T) * np.tan(Δ / 2.0))
expected_num_gates = ((3.0 - np.cos(Δ)) / np.sin(Δ)) * l1_norm_over_time(T)
print(f"L={L}, nq={nq}, Δ={Δ:.3e}, overhead≈{overhead:.3g}, E[#gates]≈{expected_num_gates:.3g}")

index = sampling.batch_sampling(np.array([probs for i in range(N)]), n_circ)
def gen_rand_cir(index):
    (gates_arr, sign, sign_list, n) = ([], 1, [], int(N / n_snap))
    p = []
    psi = QuantumState(nq)
    psi.set_computational_basis(int("00001111", 2))
    for i, inde in enumerate(index):
        circ = QuantumCircuit(nq)
        if i % n == 0:
            gates_arr.append([])
            sign_list.append(sign)
            p.append([])
            for q in range(nq):
                obs = Observable(nq)
                obs.add_operator(1.0, f"Z {q}")
                p[-1].append(float(((obs.get_expectation_value(psi)+1)/2).real))
        for j, val in inde:
            (ind, pauli) = terms[j]
            coef = coeffs[j]
            if val == 3:
                sign *= -1
                circ.add_gate(PauliRotation(ind, pauli, np.pi))
            else:
                circ.add_gate(PauliRotation(ind, pauli, np.sign(coef) * Δ))
        circ.update_quantum_state(psi)
    sign_list.append(sign)
    p.append([])
    for q in range(nq):
        obs = Observable(nq)
        obs.add_operator(1.0, f"Z {q}")
        p[-1].append(float(((obs.get_expectation_value(psi)+1)/2).real))
    return [(sign_list[i] * gam_list[i], p[i]) for i in range(n_snap + 1)]
res = []
for i in range(n_circ):
    res.append(gen_rand_cir(index[i]))
print(len(res), "circuits done")

def resample(res):
    s = np.concatenate([c * (2 * binom.rvs(1, p, size=100) - 1) for (c, p) in res])
    choices = np.reshape(s[np.random.choice(len(s), 1000 * 10000)], (10000, 1000))
    return np.mean(choices, axis=1)

qubit_n_samples = []
for q in range(nq):
    res_s_q = [(res[c][-1][0], res[c][-1][1][q]) for c in range(len(res))]
    z_samples = resample(res_s_q)
    n_samples = (1.0 - np.mean(z_samples)) / 2.0
    qubit_n_samples.append(n_samples)
for i in range(nq // 2):
    site_samples = qubit_n_samples[2*i] + qubit_n_samples[2*i+1]
    print(f"  site {i}: n_i = {site_samples}")    

L=185, nq=8, Δ=1.227e-02, overhead≈1.09, E[#gates]≈1.15e+03
10000 circuits done
  site 0: n_i = 1.9284388090332938
  site 1: n_i = 1.9085791419511058
  site 2: n_i = 0.0899886553143211
  site 3: n_i = 0.06902503724743164


In [ ]:
site 0: n_i = 1.929962
site 1: n_i = 1.911617
site 2: n_i = 0.089763
site 3: n_i = 0.068674